# <div style="text-align:center; border-radius:18px; padding:18px; margin-bottom:12px; font-size:150%; font-family:'Arial Black', Arial, sans-serif; text-shadow:2px 2px 6px rgba(0,0,0,0.7); background:linear-gradient(90deg, #1e3c72, #c31432); box-shadow:0 3px 8px rgba(0,0,0,0.35); color:white;"><b>Deep Past Challenge — Ensemble × MBR</b></div>

<a id="top"></a>

## Navigation

* [1. Setup](#section1)
* [2. Configuration](#section2)
* [3. Text Processing](#section3)
* [4. Dataset and Batching](#section4)
* [5. Model Wrapper](#section5)
* [6. MBR Selection](#section6)
* [7. Inference Pipeline](#section7)
* [8. Run and Save Submission](#section8)

# <div style="text-align:center; border-radius:15px; padding:15px; margin-top:10px; font-size:110%; font-family:'Arial Black', Arial, sans-serif; text-shadow:2px 2px 5px rgba(0,0,0,0.7); background:linear-gradient(90deg, #1e3c72, #c31432); box-shadow:0 2px 5px rgba(0,0,0,0.3); color:white;"><b>Thanks https://www.kaggle.com/meenalsinha for this amazing notebook, its impressive work :)</b></div>

<a id="section1"></a>

# <div style="text-align:center; border-radius:15px; padding:15px; font-size:115%; font-family:'Arial Black', Arial, sans-serif; text-shadow:2px 2px 5px rgba(0,0,0,0.7); background:linear-gradient(90deg, #1e3c72, #c31432); box-shadow:0 2px 5px rgba(0,0,0,0.3); color:white;"><b>1. Setup</b></div>

In [1]:
!pip install sacrebleu

In [2]:
import os
import gc
import re
import json
import math
import random
import logging
import warnings

from pathlib import Path
from dataclasses import dataclass
from contextlib import nullcontext
from typing import List, Tuple

import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, Sampler
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm

import sacrebleu

warnings.filterwarnings("ignore")

os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")

def _cuda_bf16_supported() -> bool:
    if not torch.cuda.is_available():
        return False
    try:
        return bool(getattr(torch.cuda, "is_bf16_supported", lambda: False)())
    except Exception:
        return False


def _bf16_ctx(device: torch.device, enabled: bool):
    if enabled and device.type == "cuda" and _cuda_bf16_supported():
        return torch.autocast(device_type="cuda", dtype=torch.bfloat16)
    return nullcontext()

def setup_logging(output_dir: str) -> logging.Logger:
    Path(output_dir).mkdir(exist_ok=True, parents=True)

    for h in logging.root.handlers[:]:
        logging.root.removeHandler(h)

    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
        handlers=[
            logging.StreamHandler(),
            logging.FileHandler(Path(output_dir) / "ensemble_mbr.log"),
        ],
    )
    return logging.getLogger("ensemble_mbr")

<a id="section2"></a>

# <div style="text-align:center; border-radius:15px; padding:15px; font-size:115%; font-family:'Arial Black', Arial, sans-serif; text-shadow:2px 2px 5px rgba(0,0,0,0.7); background:linear-gradient(90deg, #1e3c72, #c31432); box-shadow:0 2px 5px rgba(0,0,0,0.3); color:white;"><b>2. Configuration</b></div>

In [3]:
@dataclass
class EnsembleMBRConfig:
    test_data_path: str = "/kaggle/input/deep-past-initiative-machine-translation/test.csv"
    output_dir: str     = "/kaggle/working/"
    model_a_path: str   = "/kaggle/input/final-byt5/byt5-akkadian-optimized-34x"
    model_b_path: str   = "/kaggle/input/models/mattiaangeli/byt5-akkadian-mbr-v2/pytorch/default/1"

    max_input_length: int = 512
    max_new_tokens: int   = 384
    batch_size: int       = 2
    num_workers: int      = 2
    num_buckets: int      = 8

    num_beam_cands: int = 5
    num_beams: int      = 10
    length_penalty: float = 1.15
    early_stopping: bool  = True
    repetition_penalty: float = 1.12

    num_sample_cands: int = 3
    mbr_top_p: float       = 0.95
    mbr_temperature: float = 0.82
    mbr_pool_cap: int      = 40

    use_mixed_precision: bool    = True
    use_better_transformer: bool = True
    use_bucket_batching: bool    = True
    use_adaptive_beams: bool     = True
    aggressive_postprocessing: bool = False
    checkpoint_freq: int = 200

    def __post_init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        Path(self.output_dir).mkdir(exist_ok=True, parents=True)

        if not torch.cuda.is_available():
            self.use_mixed_precision = False
            self.use_better_transformer = False

        self.use_bf16_amp = bool(
            self.use_mixed_precision
            and self.device.type == "cuda"
            and _cuda_bf16_supported()
        )

        assert self.num_beams >= self.num_beam_cands


def print_env(cfg: EnsembleMBRConfig):
    print(f"PyTorch  : {torch.__version__}")
    print(f"CUDA     : {torch.cuda.is_available()}")

    if torch.cuda.is_available():
        print(f"GPU      : {torch.cuda.get_device_name(0)}")
        mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU Mem  : {mem:.2f} GB")
        print(f"BF16     : {_cuda_bf16_supported()}")

    print(f"BF16 AMP : {cfg.use_bf16_amp}")

<a id="section3"></a>

# <div style="text-align:center; border-radius:15px; padding:15px; font-size:115%; font-family:'Arial Black', Arial, sans-serif; text-shadow:2px 2px 5px rgba(0,0,0,0.7); background:linear-gradient(90deg, #1e3c72, #c31432); box-shadow:0 2px 5px rgba(0,0,0,0.3); color:white;"><b>3. Text Processing</b></div>

In [4]:
_V2 = re.compile(r"([aAeEiIuU])(?:2|₂)")
_V3 = re.compile(r"([aAeEiIuU])(?:3|₃)")

_ACUTE = str.maketrans({"a":"á","e":"é","i":"í","u":"ú","A":"Á","E":"É","I":"Í","U":"Ú"})
_GRAVE = str.maketrans({"a":"à","e":"è","i":"ì","u":"ù","A":"À","E":"È","I":"Ì","U":"Ù"})


def _ascii_to_diacritics(s):

    s = s.replace("sz","š").replace("SZ","Š")
    s = s.replace("s,","ṣ").replace("S,","Ṣ")
    s = s.replace("t,","ṭ").replace("T,","Ṭ")

    s = _V2.sub(lambda m: m.group(1).translate(_ACUTE), s)
    s = _V3.sub(lambda m: m.group(1).translate(_GRAVE), s)

    return s

_WS_RE = re.compile(r"\s+")

_GAP_UNIFIED_RE = re.compile(
    r"<\s*gap\s*>"
    r"|\.{3,}"
    r"|…+"
    r"|\[\.+\]"
    r"|\bx\b",
    re.I
)


def _normalize_gaps_vec(ser):
    return ser.str.replace(_GAP_UNIFIED_RE,"<gap>",regex=True)

class OptimizedPreprocessor:

    def preprocess_batch(self, texts):

        ser = pd.Series(texts).fillna("").astype(str)

        ser = ser.apply(_ascii_to_diacritics)

        ser = _normalize_gaps_vec(ser)

        ser = ser.str.replace(_WS_RE," ",regex=True)

        return ser.str.strip().tolist()

class VectorizedPostprocessor:

    def postprocess_batch(self, texts):

        s = pd.Series(texts).fillna("").astype(str)

        s = s.str.replace(_WS_RE," ",regex=True)

        return s.str.strip().tolist()

<a id="section4"></a>

# <div style="text-align:center; border-radius:15px; padding:15px; font-size:115%; font-family:'Arial Black', Arial, sans-serif; text-shadow:2px 2px 5px rgba(0,0,0,0.7); background:linear-gradient(90deg, #1e3c72, #c31432); box-shadow:0 2px 5px rgba(0,0,0,0.3); color:white;"><b>4. Dataset and Batching</b></div>

In [5]:
class AkkadianDataset(Dataset):

    def __init__(self, df, preprocessor, logger):

        self.sample_ids = df["id"].tolist()

        proc = preprocessor.preprocess_batch(df["transliteration"].tolist())

        self.input_texts = [
            "translate Akkadian to English: " + t
            for t in proc
        ]

        logger.info(f"Dataset size: {len(self.sample_ids)}")

    def __len__(self):
        return len(self.sample_ids)

    def __getitem__(self, idx):
        return self.sample_ids[idx], self.input_texts[idx]

<a id="section5"></a>

# <div style="text-align:center; border-radius:15px; padding:15px; font-size:115%; font-family:'Arial Black', Arial, sans-serif; text-shadow:2px 2px 5px rgba(0,0,0,0.7); background:linear-gradient(90deg, #1e3c72, #c31432); box-shadow:0 2px 5px rgba(0,0,0,0.3); color:white;"><b>5. Model Wrapper</b></div>

In [6]:
class ModelWrapper:

    def __init__(self, model_path, cfg, logger, label):

        self.cfg = cfg
        self.logger = logger
        self.label = label

        logger.info(f"Loading {label}")

        self.tokenizer = AutoTokenizer.from_pretrained(model_path)

        self.model = (
            AutoModelForSeq2SeqLM
            .from_pretrained(model_path)
            .to(cfg.device)
            .eval()
        )

    def collate(self, batch):

        ids = [b[0] for b in batch]
        texts = [b[1] for b in batch]

        enc = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=self.cfg.max_input_length,
            return_tensors="pt"
        )

        return ids, enc


    def generate_candidates(self, input_ids, mask):

        cfg = self.cfg
        B = input_ids.shape[0]

        ctx = _bf16_ctx(cfg.device, cfg.use_bf16_amp)

        with ctx:

            beam_out = self.model.generate(
                input_ids=input_ids,
                attention_mask=mask,
                num_beams=cfg.num_beams,
                num_return_sequences=cfg.num_beam_cands,
                do_sample=False,
                max_new_tokens=cfg.max_new_tokens,
                length_penalty=cfg.length_penalty,
                repetition_penalty=cfg.repetition_penalty
            )

            beam_texts = self.tokenizer.batch_decode(
                beam_out,
                skip_special_tokens=True
            )

            samp_out = self.model.generate(
                input_ids=input_ids,
                attention_mask=mask,
                do_sample=True,
                num_return_sequences=cfg.num_sample_cands,
                top_p=cfg.mbr_top_p,
                temperature=cfg.mbr_temperature,
                max_new_tokens=cfg.max_new_tokens
            )

            samp_texts = self.tokenizer.batch_decode(
                samp_out,
                skip_special_tokens=True
            )

        Rb = cfg.num_beam_cands
        Rs = cfg.num_sample_cands

        pools = []

        for i in range(B):
            pool = list(beam_texts[i * Rb:(i + 1) * Rb])
            pool += list(samp_texts[i * Rs:(i + 1) * Rs])
            pools.append(pool)

        return pools
            

<a id="section6"></a>

# <div style="text-align:center; border-radius:15px; padding:15px; font-size:115%; font-family:'Arial Black', Arial, sans-serif; text-shadow:2px 2px 5px rgba(0,0,0,0.7); background:linear-gradient(90deg, #1e3c72, #c31432); box-shadow:0 2px 5px rgba(0,0,0,0.3); color:white;"><b>6. MBR Selection</b></div>

In [7]:
class MBRSelector:

    def __init__(self,pool_cap=40):

        self.metric=sacrebleu.metrics.CHRF(word_order=2)

        self.pool_cap=pool_cap


    def _score(self,a,b):

        if not a or not b:
            return 0

        return self.metric.sentence_score(a,[b]).score


    def pick(self,candidates):

        candidates=list(dict.fromkeys(candidates))

        candidates=candidates[:self.pool_cap]

        if len(candidates)==1:
            return candidates[0]

        best=None
        best_score=-1

        for i,a in enumerate(candidates):

            score=0

            for j,b in enumerate(candidates):

                if i!=j:
                    score+=self._score(a,b)

            score/=max(1,len(candidates)-1)

            if score>best_score:
                best_score=score
                best=a

        return best

<a id="section7"></a>

# <div style="text-align:center; border-radius:15px; padding:15px; font-size:115%; font-family:'Arial Black', Arial, sans-serif; text-shadow:2px 2px 5px rgba(0,0,0,0.7); background:linear-gradient(90deg, #1e3c72, #c31432); box-shadow:0 2px 5px rgba(0,0,0,0.3); color:white;"><b>7. Inference Pipeline</b></div>

In [8]:
class EnsembleEngine:

    def __init__(self, cfg, logger):

        self.cfg = cfg
        self.logger = logger

        self.preprocessor = OptimizedPreprocessor()
        self.postprocessor = VectorizedPostprocessor()

        self.mbr = MBRSelector(cfg.mbr_pool_cap)


    def run(self, test_df):

        dataset = AkkadianDataset(
            test_df,
            self.preprocessor,
            self.logger
        )

        wrapperA = ModelWrapper(
            self.cfg.model_a_path,
            self.cfg,
            self.logger,
            "Model-A"
        )

        wrapperB = ModelWrapper(
            self.cfg.model_b_path,
            self.cfg,
            self.logger,
            "Model-B"
        )

        loader = DataLoader(
            dataset,
            batch_size=self.cfg.batch_size,
            shuffle=False,
            collate_fn=wrapperA.collate
        )

        results = []

        for ids, enc in tqdm(loader):

            input_ids = enc.input_ids.to(self.cfg.device)
            mask = enc.attention_mask.to(self.cfg.device)

            batch_pools_A = wrapperA.generate_candidates(input_ids, mask)
            batch_pools_B = wrapperB.generate_candidates(input_ids, mask)

            for sid, poolA, poolB in zip(ids, batch_pools_A, batch_pools_B):

                pool = poolA + poolB
                pool = self.postprocessor.postprocess_batch(pool)

                best = self.mbr.pick(pool)

                if not best or not str(best).strip():
                    best = "The tablet is too damaged to translate."

                results.append(best)

        return results

<a id="section8"></a>

# <div style="text-align:center; border-radius:15px; padding:15px; font-size:115%; font-family:'Arial Black', Arial, sans-serif; text-shadow:2px 2px 5px rgba(0,0,0,0.7); background:linear-gradient(90deg, #1e3c72, #c31432); box-shadow:0 2px 5px rgba(0,0,0,0.3); color:white;"><b>8. Run and Save Submission</b></div>

In [9]:
cfg = EnsembleMBRConfig()

logger = setup_logging(cfg.output_dir)

print("Device:", cfg.device)

test_df = pd.read_csv(cfg.test_data_path)

print("Test size:", len(test_df))

engine = EnsembleEngine(cfg, logger)
translations = engine.run(test_df)

submission = pd.DataFrame({
    "id": test_df["id"],
    "translation": translations
})

output_path = "/kaggle/working/submission.csv"
submission.to_csv(output_path, index=False)

print("\nSubmission saved to:", output_path)

import os
print("File exists:", os.path.exists(output_path))

print("\nFiles in /kaggle/working:")
print(os.listdir("/kaggle/working"))

print("\nSubmission preview:")
print(submission.head())

2026-03-09 00:38:42,662 | INFO | Dataset size: 4
2026-03-09 00:38:42,663 | INFO | Loading Model-A


Device: cuda
Test size: 4


2026-03-09 00:38:45.345637: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773016725.542662      36 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773016725.603491      36 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773016726.117912      36 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773016726.117961      36 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773016726.117964      36 computation_placer.cc:177] computation placer alr

  0%|          | 0/2 [00:00<?, ?it/s]


Submission saved to: /kaggle/working/submission.csv
File exists: True

Files in /kaggle/working:
['__notebook__.ipynb', 'ensemble_mbr.log', 'submission.csv']

Submission preview:
   id                                        translation
0   0  Thus says the Kanesh colony: Speak to our mess...
1   1  As for the tablet of the City, whoever buys me...
2   2  As soon as you have heard our letter, whether ...
3   3  I sent it to every single colony and the tradi...
